# Citi Bike trip data → Parquet

Download monthly trip data from the [Citi Bike public archive](https://s3.amazonaws.com/tripdata/index.html) and combine it into one compressed Parquet file.

- **Period:** August 2025–August 2026 (inclusive).
- **Output:** `citibike_2025_2026_data.parquet`, beside this notebook.
- **Usage:** Keep the notebook name `Download_Data_clean.ipynb`, start it with its containing folder as the working directory, and run all cells in order. Python 3.10+ is required.

Downloads require internet access and sufficient temporary disk space for the extracted CSV files. Temporary files are removed automatically. A successful run replaces any existing output file. No account credentials or cloud storage are required.

In [ ]:
# Install the only external dependency in this notebook's environment.
%pip install -q duckdb

In [ ]:
from pathlib import Path
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

import duckdb

BASE_URL = "https://s3.amazonaws.com/tripdata/"
MONTHS = [f"2025{month:02d}" for month in range(8, 13)] + [
    f"2026{month:02d}" for month in range(1, 9)
]

# Verify the working directory so the result stays beside this notebook.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "Download_Data_clean.ipynb").is_file():
    raise RuntimeError("Start this notebook from the folder containing Download_Data_clean.ipynb.")

OUTPUT_PATH = NOTEBOOK_DIR / "citibike_2025_2026_data.parquet"

In [ ]:
# Keep downloads and conversion files temporary, on the same disk as the output.
with TemporaryDirectory(prefix="citibike_", dir=NOTEBOOK_DIR) as temp_dir:
    work_dir = Path(temp_dir)
    csv_paths = []

    for month in MONTHS:
        filename = f"{month}-citibike-tripdata.zip"
        archive_path = work_dir / filename
        print(f"Downloading {filename}...")
        urlretrieve(BASE_URL + filename, archive_path)

        # Read CSVs even when they are inside a folder within the ZIP.
        with ZipFile(archive_path) as archive:
            members = [
                member for member in archive.infolist()
                if member.filename.lower().endswith(".csv")
                and "__MACOSX" not in Path(member.filename).parts
                and not Path(member.filename).name.startswith("._")
            ]
            if not members:
                raise ValueError(f"No CSV files found in {filename}.")
            for index, member in enumerate(members):
                csv_path = work_dir / f"{month}_{index}.csv"
                with archive.open(member) as source, csv_path.open("wb") as target:
                    copyfileobj(source, target)
                csv_paths.append(str(csv_path))
        archive_path.unlink()

    print("Combining CSV files and writing Parquet...")
    temporary_output = work_dir / OUTPUT_PATH.name
    with duckdb.connect() as connection:
        # Allow disk spilling and preserve station IDs as text.
        connection.execute("SET temp_directory = ?", [str(work_dir / "duckdb_tmp")])
        connection.read_csv(
            csv_paths,
            union_by_name=True,
            dtype={"start_station_id": "VARCHAR", "end_station_id": "VARCHAR"},
        ).write_parquet(str(temporary_output), compression="zstd")

    # Replace the final file only after conversion succeeds.
    temporary_output.replace(OUTPUT_PATH)

print(f"Saved: {OUTPUT_PATH}")